In [7]:
# ============================================================
# CELL 1 — Imports
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_fscore_support,
    f1_score,
)

RANDOM_STATE = 42

print("Imports complete.")

Imports complete.


In [8]:
# ============================================================
# CELL 2 — Load CSV parts
# ============================================================

CSV_PARTS = [
    'perf_metrics1.csv',
    'perf_metrics2.csv',
]

dfs = []

for path in CSV_PARTS:

    if not os.path.exists(path):
        print(f"WARNING: not found -> {path}")
        continue

    part = pd.read_csv(path)
    part['source_file'] = path

    dfs.append(part)

    print(f"Loaded {path:<20}: {len(part):>8} rows")

if not dfs:
    raise FileNotFoundError(
        "Neither perf_metrics1.csv nor perf_metrics2.csv was found."
    )

df = pd.concat(dfs, ignore_index=True)

print("\nCombined shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Loaded perf_metrics1.csv   :   526895 rows
Loaded perf_metrics2.csv   :   250683 rows

Combined shape: (777578, 45)

Columns:
['timestamp_ns', 'pid', 'cpu', 'comm', 'ctx_switches', 'voluntary_switches', 'involuntary_switches', 'cpu_migrations', 'total_runtime_ns', 'stall_ns', 'avg_stall_ns', 'max_stall_ns', 'latency_count', 'avg_runq_ratio', 'minor_faults', 'major_faults', 'kmalloc_count', 'kfree_count', 'total_alloc_bytes', 'total_free_bytes', 'large_page_allocs', 'syscall_count', 'avg_syscall_latency_ns', 'max_syscall_latency_ns', 'read_count', 'write_count', 'read_bytes', 'write_bytes', 'mmap_count', 'futex_count', 'avg_futex_latency_ns', 'epoll_count', 'avg_epoll_latency_ns', 'poll_count', 'syscall_error_count', 'mutex_contentions', 'avg_mutex_wait_ns', 'max_mutex_wait_ns', 'rwsem_read_contentions', 'avg_rwsem_read_wait_ns', 'rwsem_write_contentions', 'avg_rwsem_write_wait_ns', 'max_rwsem_write_wait_ns', 'session_label', 'source_file']


In [9]:
# ============================================================
# CELL 3 — Basic checks
# ============================================================

if 'avg_stall_ns' not in df.columns:
    raise KeyError(
        "'avg_stall_ns' was not found in the CSV files."
    )

print("avg_stall_ns dtype:", df['avg_stall_ns'].dtype)
print("Missing avg_stall_ns:", df['avg_stall_ns'].isna().sum())
print("Infinite avg_stall_ns:",
      np.isinf(pd.to_numeric(df['avg_stall_ns'], errors='coerce')).sum())

print("\nDataset shape:", df.shape)

display(df.head())

avg_stall_ns dtype: int64
Missing avg_stall_ns: 0
Infinite avg_stall_ns: 0

Dataset shape: (777578, 45)


,timestamp_ns,pid,cpu,comm,ctx_switches,voluntary_switches,involuntary_switches,cpu_migrations,total_runtime_ns,stall_ns,...,mutex_contentions,avg_mutex_wait_ns,max_mutex_wait_ns,rwsem_read_contentions,avg_rwsem_read_wait_ns,rwsem_write_contentions,avg_rwsem_write_wait_ns,max_rwsem_write_wait_ns,session_label,source_file
0,15253211214479,53292,4,TaskCon~ller #2,9,9,0,1,256663,57817,...,0,0,0,0,0,0,0,0,idle,perf_metrics1.csv
1,15326325471859,158069,2,docker,10,9,1,1,3143651,12296,...,1,982,982,0,0,0,0,0,idle,perf_metrics1.csv
2,15200712630438,157330,6,docker,6,6,0,0,1642845,49874,...,0,0,0,0,0,0,0,0,idle,perf_metrics1.csv
3,15281749341127,157801,6,docker,3,3,0,0,634617,47731,...,0,0,0,0,0,0,0,0,idle,perf_metrics1.csv
4,15299979404987,157910,10,docker,1,0,1,1,38151,1563,...,0,0,0,0,0,0,0,0,idle,perf_metrics1.csv


In [10]:
# ============================================================
# CELL 4 — Clean target measurement
# ============================================================

df['avg_stall_ns'] = pd.to_numeric(
    df['avg_stall_ns'],
    errors='coerce'
)

# Remove rows where the actual latency measurement is unusable
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['avg_stall_ns']).reset_index(drop=True)

print("Shape after cleaning:", df.shape)

print("\navg_stall_ns summary:")
display(df['avg_stall_ns'].describe())

Shape after cleaning: (777578, 45)

avg_stall_ns summary:


count    7.775780e+05
mean     3.751545e+05
std      4.493643e+06
min      0.000000e+00
25%      3.787000e+03
50%      8.894700e+04
75%      3.700552e+05
max      2.612931e+09
Name: avg_stall_ns, dtype: float64

In [11]:
# ============================================================
# CELL 5 — Feature selection
# ============================================================

TARGET_MEASUREMENT = 'avg_stall_ns'

EXCLUDE_COLUMNS = {
    TARGET_MEASUREMENT,
    'session_label',
    'source_file',
}

numeric_columns = df.select_dtypes(
    include=[np.number]
).columns.tolist()

FEATURE_COLS = [
    col for col in numeric_columns
    if col not in EXCLUDE_COLUMNS
]

if not FEATURE_COLS:
    raise ValueError(
        "No numeric feature columns remain after exclusions."
    )

print("Number of numeric features:", len(FEATURE_COLS))

print("\nFeatures:")
for col in FEATURE_COLS:
    print(" -", col)

Number of numeric features: 41

Features:
 - timestamp_ns
 - pid
 - cpu
 - ctx_switches
 - voluntary_switches
 - involuntary_switches
 - cpu_migrations
 - total_runtime_ns
 - stall_ns
 - max_stall_ns
 - latency_count
 - avg_runq_ratio
 - minor_faults
 - major_faults
 - kmalloc_count
 - kfree_count
 - total_alloc_bytes
 - total_free_bytes
 - large_page_allocs
 - syscall_count
 - avg_syscall_latency_ns
 - max_syscall_latency_ns
 - read_count
 - write_count
 - read_bytes
 - write_bytes
 - mmap_count
 - futex_count
 - avg_futex_latency_ns
 - epoll_count
 - avg_epoll_latency_ns
 - poll_count
 - syscall_error_count
 - mutex_contentions
 - avg_mutex_wait_ns
 - max_mutex_wait_ns
 - rwsem_read_contentions
 - avg_rwsem_read_wait_ns
 - rwsem_write_contentions
 - avg_rwsem_write_wait_ns
 - max_rwsem_write_wait_ns


In [12]:
# ============================================================
# CELL 6 — Remove obvious metadata / ID columns if necessary
# ============================================================

# Add columns here ONLY if they are identifiers or metadata,
# rather than genuine performance measurements.

MANUAL_EXCLUDE = [
    # Example:
    # 'pid',
    # 'timestamp',
    # 'row_id',
]

FEATURE_COLS = [
    col for col in FEATURE_COLS
    if col not in MANUAL_EXCLUDE
]

print("Final feature count:", len(FEATURE_COLS))
print(FEATURE_COLS)

Final feature count: 41
['timestamp_ns', 'pid', 'cpu', 'ctx_switches', 'voluntary_switches', 'involuntary_switches', 'cpu_migrations', 'total_runtime_ns', 'stall_ns', 'max_stall_ns', 'latency_count', 'avg_runq_ratio', 'minor_faults', 'major_faults', 'kmalloc_count', 'kfree_count', 'total_alloc_bytes', 'total_free_bytes', 'large_page_allocs', 'syscall_count', 'avg_syscall_latency_ns', 'max_syscall_latency_ns', 'read_count', 'write_count', 'read_bytes', 'write_bytes', 'mmap_count', 'futex_count', 'avg_futex_latency_ns', 'epoll_count', 'avg_epoll_latency_ns', 'poll_count', 'syscall_error_count', 'mutex_contentions', 'avg_mutex_wait_ns', 'max_mutex_wait_ns', 'rwsem_read_contentions', 'avg_rwsem_read_wait_ns', 'rwsem_write_contentions', 'avg_rwsem_write_wait_ns', 'max_rwsem_write_wait_ns']


In [13]:
# ============================================================
# CELL 7 — Prepare model matrix
# ============================================================

X = (
    df[FEATURE_COLS]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype('float32')
)

stall_ns = df['avg_stall_ns'].to_numpy(dtype=np.float64)

print("X shape:", X.shape)
print("Latency values:", len(stall_ns))

X shape: (777578, 41)
Latency values: 777578


In [14]:
# ============================================================
# CELL 8 — Candidate latency boundaries
# ============================================================

NORMAL_CANDIDATES_MS = [
    0.025,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.20,
    0.25,
    0.30,
    0.50,
]

HIGH_CANDIDATES_MS = [
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    2.00,
    3.00,
    5.00,
]

print("Normal candidates:", NORMAL_CANDIDATES_MS)
print("High candidates  :", HIGH_CANDIDATES_MS)

Normal candidates: [0.025, 0.05, 0.075, 0.1, 0.125, 0.15, 0.2, 0.25, 0.3, 0.5]
High candidates  : [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 5.0]


In [15]:
# ============================================================
# CELL 9 — Fixed train/test split
# ============================================================

all_indices = np.arange(len(df))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True
)

X_train = X.iloc[train_indices]
X_test = X.iloc[test_indices]

print("Training rows:", len(train_indices))
print("Testing rows :", len(test_indices))

Training rows: 622062
Testing rows : 155516


In [16]:
# ============================================================
# CELL 10 — Label generation function
# ============================================================

def make_labels(stall_ns, normal_ms, high_ms):
    """
    Create:
        0 = Normal
        1 = Medium
        2 = High

    based ONLY on avg_stall_ns.
    """

    normal_ns = normal_ms * 1e6
    high_ns = high_ms * 1e6

    if normal_ns >= high_ns:
        raise ValueError(
            f"Normal threshold ({normal_ms} ms) must be "
            f"below High threshold ({high_ms} ms)."
        )

    return np.select(
        [
            stall_ns < normal_ns,
            stall_ns < high_ns,
        ],
        [
            0,
            1,
        ],
        default=2
    )

In [17]:
# ============================================================
# CELL 11 — Full threshold sweep
# ============================================================

N_ESTIMATORS_SWEEP = 200

threshold_results = []

total_configs = sum(
    normal_ms < high_ms
    for normal_ms in NORMAL_CANDIDATES_MS
    for high_ms in HIGH_CANDIDATES_MS
)

config_number = 0

for normal_ms in NORMAL_CANDIDATES_MS:

    for high_ms in HIGH_CANDIDATES_MS:

        if normal_ms >= high_ms:
            continue

        config_number += 1

        print(
            f"\rRunning {config_number}/{total_configs}: "
            f"Normal={normal_ms:.3f} ms, "
            f"High={high_ms:.3f} ms",
            end=''
        )

        # ----------------------------------------------------
        # Create labels
        # ----------------------------------------------------

        y_all = make_labels(
            stall_ns,
            normal_ms,
            high_ms
        )

        y_train = y_all[train_indices]
        y_test = y_all[test_indices]

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        model = RandomForestClassifier(
            n_estimators=N_ESTIMATORS_SWEEP,
            min_samples_leaf=5,
            class_weight='balanced',
            n_jobs=-1,
            random_state=RANDOM_STATE
        )

        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        precision, recall, f1, support = (
            precision_recall_fscore_support(
                y_test,
                pred,
                labels=[0, 1, 2],
                zero_division=0
            )
        )

        cm = confusion_matrix(
            y_test,
            pred,
            labels=[0, 1, 2]
        )

        threshold_results.append({

            # Thresholds
            'normal_ms': normal_ms,
            'high_ms': high_ms,

            # Class distribution
            'normal_n': int((y_all == 0).sum()),
            'medium_n': int((y_all == 1).sum()),
            'high_n': int((y_all == 2).sum()),
            'high_pct': 100 * (y_all == 2).mean(),

            # Overall
            'accuracy': accuracy_score(y_test, pred),
            'macro_f1': f1_score(
                y_test,
                pred,
                average='macro',
                zero_division=0
            ),

            # HIGH — most important
            'high_precision': precision[2],
            'high_recall': recall[2],
            'high_f1': f1[2],

            # High false negatives
            'high_to_normal': int(cm[2, 0]),
            'high_to_medium': int(cm[2, 1]),

            # False High predictions
            'normal_to_high': int(cm[0, 2]),
            'medium_to_high': int(cm[1, 2]),
        })

print("\n\nThreshold sweep complete.")

threshold_results = pd.DataFrame(threshold_results)

print("Configurations tested:", len(threshold_results))

Running 86/86: Normal=0.500 ms, High=5.000 ms

Threshold sweep complete.
Configurations tested: 86


In [18]:
# ============================================================
# CELL 12 — All threshold results
# ============================================================

display(
    threshold_results
    .sort_values(
        ['high_f1', 'high_recall'],
        ascending=False
    )
    .round(4)
)

,normal_ms,high_ms,normal_n,medium_n,high_n,high_pct,accuracy,macro_f1,high_precision,high_recall,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high
0,0.025,0.25,306158,200411,271009,34.8530,0.9864,0.9850,0.9907,0.9848,0.9878,0,823,1,499
9,0.050,0.25,345560,161009,271009,34.8530,0.9830,0.9794,0.9929,0.9807,0.9868,1,1048,1,379
18,0.075,0.25,374869,131700,271009,34.8530,0.9785,0.9710,0.9941,0.9763,0.9851,0,1288,2,314
27,0.100,0.25,398767,107802,271009,34.8530,0.9749,0.9621,0.9951,0.9718,0.9833,0,1529,4,258
36,0.125,0.25,419602,86967,271009,34.8530,0.9689,0.9468,0.9953,0.9662,0.9805,2,1834,6,240
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44,0.125,5.00,419602,348998,8978,1.1546,0.9896,0.9531,0.7901,0.9865,0.8775,0,25,0,487
35,0.100,5.00,398767,369833,8978,1.1546,0.9902,0.9521,0.7803,0.9903,0.8729,0,18,0,518
26,0.075,5.00,374869,393731,8978,1.1546,0.9910,0.9522,0.7793,0.9882,0.8714,0,22,0,520
17,0.050,5.00,345560,423040,8978,1.1546,0.9915,0.9519,0.7746,0.9898,0.8691,0,19,0,535


In [19]:
# ============================================================
# CELL 13 — Rank by HIGH RECALL
# ============================================================

top_recall = (
    threshold_results
    .sort_values(
        ['high_recall', 'high_f1'],
        ascending=False
    )
    .head(20)
)

display(
    top_recall[
        [
            'normal_ms',
            'high_ms',
            'high_pct',
            'high_recall',
            'high_precision',
            'high_f1',
            'high_to_normal',
            'high_to_medium',
            'normal_to_high',
            'medium_to_high',
            'accuracy',
            'macro_f1'
        ]
    ].round(4)
)

,normal_ms,high_ms,high_pct,high_recall,high_precision,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high,accuracy,macro_f1
6,0.025,2.00,3.1265,0.9969,0.8925,0.9418,0,15,0,590,0.9917,0.9763
15,0.050,2.00,3.1265,0.9967,0.9001,0.9460,0,16,0,543,0.9912,0.9772
24,0.075,2.00,3.1265,0.9965,0.9040,0.9480,0,17,0,520,0.9906,0.9774
42,0.125,2.00,3.1265,0.9959,0.9115,0.9518,0,20,0,475,0.9892,0.9774
33,0.100,2.00,3.1265,0.9959,0.9064,0.9491,0,20,0,505,0.9897,0.9770
51,0.150,2.00,3.1265,0.9951,0.9153,0.9536,0,24,0,452,0.9878,0.9768
60,0.200,2.00,3.1265,0.9951,0.9152,0.9535,0,24,0,453,0.9845,0.9739
16,0.050,3.00,2.3518,0.9949,0.8724,0.9296,0,19,0,539,0.9909,0.9716
68,0.250,2.00,3.1265,0.9947,0.9212,0.9565,0,26,0,418,0.9824,0.9726
5,0.025,1.50,3.7043,0.9946,0.9050,0.9477,0,31,0,602,0.9914,0.9781


In [20]:
# ============================================================
# CELL 14 — Rank by HIGH F1
# ============================================================

top_f1 = (
    threshold_results
    .sort_values(
        ['high_f1', 'high_recall'],
        ascending=False
    )
    .head(20)
)

display(
    top_f1[
        [
            'normal_ms',
            'high_ms',
            'high_pct',
            'high_recall',
            'high_precision',
            'high_f1',
            'high_to_normal',
            'high_to_medium',
            'normal_to_high',
            'medium_to_high',
            'accuracy',
            'macro_f1'
        ]
    ].round(4)
)

,normal_ms,high_ms,high_pct,high_recall,high_precision,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high,accuracy,macro_f1
0,0.025,0.25,34.8530,0.9848,0.9907,0.9878,0,823,1,499,0.9864,0.9850
9,0.050,0.25,34.8530,0.9807,0.9929,0.9868,1,1048,1,379,0.9830,0.9794
18,0.075,0.25,34.8530,0.9763,0.9941,0.9851,0,1288,2,314,0.9785,0.9710
27,0.100,0.25,34.8530,0.9718,0.9951,0.9833,0,1529,4,258,0.9749,0.9621
36,0.125,0.25,34.8530,0.9662,0.9953,0.9805,2,1834,6,240,0.9689,0.9468
45,0.150,0.25,34.8530,0.9610,0.9954,0.9779,6,2108,19,220,0.9633,0.9276
54,0.200,0.25,34.8530,0.9588,0.9921,0.9752,25,2210,137,275,0.9575,0.8710
82,0.500,1.50,3.7043,0.9849,0.9629,0.9738,0,87,0,219,0.9705,0.9443
81,0.500,1.25,4.1465,0.9804,0.9651,0.9727,0,127,0,230,0.9691,0.9402
80,0.500,1.00,4.7854,0.9796,0.9655,0.9725,0,152,7,253,0.9667,0.9340


In [21]:
# ============================================================
# CELL 15 — Reasonable High-class size
# ============================================================

reasonable = threshold_results[
    (threshold_results['high_pct'] >= 2.0) &
    (threshold_results['high_pct'] <= 10.0)
].copy()

print(
    "Candidates remaining:",
    len(reasonable)
)

display(
    reasonable
    .sort_values(
        ['high_f1', 'high_recall'],
        ascending=False
    )
    [
        [
            'normal_ms',
            'high_ms',
            'high_pct',
            'high_recall',
            'high_precision',
            'high_f1',
            'high_to_normal',
            'high_to_medium',
            'normal_to_high',
            'medium_to_high',
            'accuracy',
            'macro_f1'
        ]
    ]
    .head(25)
    .round(4)
)

Candidates remaining: 60


,normal_ms,high_ms,high_pct,high_recall,high_precision,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high,accuracy,macro_f1
82,0.500,1.50,3.7043,0.9849,0.9629,0.9738,0,87,0,219,0.9705,0.9443
81,0.500,1.25,4.1465,0.9804,0.9651,0.9727,0,127,0,230,0.9691,0.9402
80,0.500,1.00,4.7854,0.9796,0.9655,0.9725,0,152,7,253,0.9667,0.9340
83,0.500,2.00,3.1265,0.9857,0.9554,0.9703,0,70,0,226,0.9720,0.9471
73,0.300,1.00,4.7854,0.9901,0.9459,0.9675,0,74,4,417,0.9781,0.9710
72,0.300,0.75,7.0493,0.9730,0.9617,0.9673,1,296,16,410,0.9752,0.9674
65,0.250,1.00,4.7854,0.9907,0.9429,0.9662,0,69,1,445,0.9803,0.9738
64,0.250,0.75,7.0493,0.9755,0.9565,0.9659,1,268,10,478,0.9784,0.9717
57,0.200,1.00,4.7854,0.9911,0.9406,0.9652,0,66,0,466,0.9834,0.9768
56,0.200,0.75,7.0493,0.9763,0.9537,0.9649,0,260,6,515,0.9806,0.9744


In [22]:
# ============================================================
# CELL 16 — Select candidate threshold pair
# ============================================================

best_row = (
    reasonable
    .sort_values(
        ['high_f1', 'high_recall'],
        ascending=False
    )
    .iloc[0]
)

BEST_NORMAL_MS = float(best_row['normal_ms'])
BEST_HIGH_MS = float(best_row['high_ms'])

print("Selected candidate:")
print(f"  Normal threshold : {BEST_NORMAL_MS:.3f} ms")
print(f"  High threshold   : {BEST_HIGH_MS:.3f} ms")
print(f"  High prevalence  : {best_row['high_pct']:.2f}%")
print(f"  High precision   : {best_row['high_precision']:.4f}")
print(f"  High recall      : {best_row['high_recall']:.4f}")
print(f"  High F1          : {best_row['high_f1']:.4f}")

Selected candidate:
  Normal threshold : 0.500 ms
  High threshold   : 1.500 ms
  High prevalence  : 3.70%
  High precision   : 0.9629
  High recall      : 0.9849
  High F1          : 0.9738


In [23]:
# ============================================================
# CELL 17 — Detailed evaluation of selected candidate
# ============================================================

y_best = make_labels(
    stall_ns,
    BEST_NORMAL_MS,
    BEST_HIGH_MS
)

y_train_best = y_best[train_indices]
y_test_best = y_best[test_indices]

final_candidate_model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

final_candidate_model.fit(
    X_train,
    y_train_best
)

pred_best = final_candidate_model.predict(X_test)

cm_best = confusion_matrix(
    y_test_best,
    pred_best,
    labels=[0, 1, 2]
)

print("Thresholds:")
print(f"Normal < {BEST_NORMAL_MS} ms")
print(f"Medium = {BEST_NORMAL_MS}–{BEST_HIGH_MS} ms")
print(f"High >= {BEST_HIGH_MS} ms")

print("\nConfusion Matrix:")
print(cm_best)

print("\nClassification Report:")
print(
    classification_report(
        y_test_best,
        pred_best,
        target_names=[
            'Normal',
            'Medium',
            'High'
        ],
        digits=4,
        zero_division=0
    )
)

Thresholds:
Normal < 0.5 ms
Medium = 0.5–1.5 ms
High >= 1.5 ms

Confusion Matrix:
[[129205   4008      0]
 [   205  16115    218]
 [     0     89   5676]]

Classification Report:
              precision    recall  f1-score   support

      Normal     0.9984    0.9699    0.9840    133213
      Medium     0.7973    0.9744    0.8770     16538
        High     0.9630    0.9846    0.9737      5765

    accuracy                         0.9709    155516
   macro avg     0.9196    0.9763    0.9449    155516
weighted avg     0.9757    0.9709    0.9722    155516



In [24]:
# ============================================================
# CELL 18 — HIGH vs NOT-HIGH
# ============================================================

actual_high = (y_test_best == 2)
predicted_high = (pred_best == 2)

high_precision, high_recall, high_f1, _ = (
    precision_recall_fscore_support(
        actual_high,
        predicted_high,
        average='binary',
        zero_division=0
    )
)

tn, fp, fn, tp = confusion_matrix(
    actual_high,
    predicted_high,
    labels=[False, True]
).ravel()

print("HIGH vs NOT-HIGH")
print("----------------")
print(f"High precision : {high_precision:.4f}")
print(f"High recall    : {high_recall:.4f}")
print(f"High F1        : {high_f1:.4f}")
print()
print(f"True negatives : {tn}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True positives : {tp}")

HIGH vs NOT-HIGH
----------------
High precision : 0.9630
High recall    : 0.9846
High F1        : 0.9737

True negatives : 149533
False positives: 218
False negatives: 89
True positives : 5676


In [25]:
BEST_NORMAL_MS
BEST_HIGH_MS

1.5

In [26]:
# ============================================================
# Inspect threshold sweep around the promising region
# ============================================================

around = threshold_results[
    (threshold_results['high_ms'] >= 1.0) &
    (threshold_results['high_ms'] <= 2.0)
].copy()

display(
    around
    .sort_values(['high_ms', 'normal_ms'])
    [
        [
            'normal_ms',
            'high_ms',
            'high_pct',
            'high_precision',
            'high_recall',
            'high_f1',
            'high_to_normal',
            'high_to_medium',
            'normal_to_high',
            'medium_to_high',
            'accuracy',
            'macro_f1'
        ]
    ]
    .round(4)
)

,normal_ms,high_ms,high_pct,high_precision,high_recall,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high,accuracy,macro_f1
3,0.025,1.00,4.7854,0.9238,0.9935,0.9574,0,48,0,610,0.9912,0.9813
12,0.050,1.00,4.7854,0.9287,0.9926,0.9596,0,55,0,567,0.9907,0.9815
21,0.075,1.00,4.7854,0.9324,0.9922,0.9614,0,58,0,535,0.9897,0.9812
30,0.100,1.00,4.7854,0.9335,0.9921,0.9619,0,59,0,526,0.9889,0.9807
39,0.125,1.00,4.7854,0.9355,0.9923,0.9631,0,57,0,509,0.9878,0.9802
48,0.150,1.00,4.7854,0.9362,0.9922,0.9634,0,58,0,503,0.9864,0.9790
57,0.200,1.00,4.7854,0.9406,0.9911,0.9652,0,66,0,466,0.9834,0.9768
65,0.250,1.00,4.7854,0.9429,0.9907,0.9662,0,69,1,445,0.9803,0.9738
73,0.300,1.00,4.7854,0.9459,0.9901,0.9675,0,74,4,417,0.9781,0.9710
80,0.500,1.00,4.7854,0.9655,0.9796,0.9725,0,152,7,253,0.9667,0.9340


In [27]:
# ============================================================
# Inspect specific HIGH thresholds
# ============================================================

specific = threshold_results[
    threshold_results['high_ms'].isin([1.0, 1.25, 1.5, 2.0])
].copy()

display(
    specific
    .sort_values(['high_ms', 'normal_ms'])
    [
        [
            'normal_ms',
            'high_ms',
            'high_pct',
            'high_precision',
            'high_recall',
            'high_f1',
            'high_to_normal',
            'high_to_medium',
            'normal_to_high',
            'medium_to_high',
            'accuracy',
            'macro_f1'
        ]
    ]
    .round(4)
)

,normal_ms,high_ms,high_pct,high_precision,high_recall,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high,accuracy,macro_f1
3,0.025,1.00,4.7854,0.9238,0.9935,0.9574,0,48,0,610,0.9912,0.9813
12,0.050,1.00,4.7854,0.9287,0.9926,0.9596,0,55,0,567,0.9907,0.9815
21,0.075,1.00,4.7854,0.9324,0.9922,0.9614,0,58,0,535,0.9897,0.9812
30,0.100,1.00,4.7854,0.9335,0.9921,0.9619,0,59,0,526,0.9889,0.9807
39,0.125,1.00,4.7854,0.9355,0.9923,0.9631,0,57,0,509,0.9878,0.9802
48,0.150,1.00,4.7854,0.9362,0.9922,0.9634,0,58,0,503,0.9864,0.9790
57,0.200,1.00,4.7854,0.9406,0.9911,0.9652,0,66,0,466,0.9834,0.9768
65,0.250,1.00,4.7854,0.9429,0.9907,0.9662,0,69,1,445,0.9803,0.9738
73,0.300,1.00,4.7854,0.9459,0.9901,0.9675,0,74,4,417,0.9781,0.9710
80,0.500,1.00,4.7854,0.9655,0.9796,0.9725,0,152,7,253,0.9667,0.9340


In [28]:
# ============================================================
# Best NORMAL threshold for each HIGH threshold
# ============================================================

best_per_high = (
    threshold_results
    .sort_values(
        ['high_ms', 'high_f1', 'high_recall'],
        ascending=[True, False, False]
    )
    .groupby('high_ms', as_index=False)
    .first()
)

display(
    best_per_high[
        [
            'normal_ms',
            'high_ms',
            'high_pct',
            'high_precision',
            'high_recall',
            'high_f1',
            'high_to_normal',
            'high_to_medium',
            'normal_to_high',
            'medium_to_high',
            'accuracy',
            'macro_f1'
        ]
    ]
    .sort_values('high_ms')
    .round(4)
)

,normal_ms,high_ms,high_pct,high_precision,high_recall,high_f1,high_to_normal,high_to_medium,normal_to_high,medium_to_high,accuracy,macro_f1
0,0.025,0.25,34.8530,0.9907,0.9848,0.9878,0,823,1,499,0.9864,0.9850
1,0.025,0.50,14.2723,0.9352,0.9731,0.9537,0,600,1,1504,0.9819,0.9761
2,0.300,0.75,7.0493,0.9617,0.9730,0.9673,1,296,16,410,0.9752,0.9674
3,0.500,1.00,4.7854,0.9655,0.9796,0.9725,0,152,7,253,0.9667,0.9340
4,0.500,1.25,4.1465,0.9651,0.9804,0.9727,0,127,0,230,0.9691,0.9402
5,0.500,1.50,3.7043,0.9629,0.9849,0.9738,0,87,0,219,0.9705,0.9443
6,0.500,2.00,3.1265,0.9554,0.9857,0.9703,0,70,0,226,0.9720,0.9471
7,0.500,3.00,2.3518,0.9322,0.9841,0.9574,0,59,0,265,0.9723,0.9453
8,0.500,5.00,1.1546,0.8805,0.9715,0.9237,0,53,0,245,0.9757,0.9414
